# K-Means Clustering Analysis

This notebook:
1. Loads and preprocesses the waste management dataset
2. Encodes all features (excluding City/District)
3. Applies K-means clustering with 3 clusters
4. Splits data into training (80%) and testing (20%)
5. Saves training data as X.npy and y.npy


In [1]:
# Import required libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')


## Step 1: Load and Preprocess Data


In [2]:
# Load the dataset
df = pd.read_csv('w.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()


Dataset shape: (850, 13)

Columns: ['City/District', 'Waste Type', 'Waste Generated (Tons/Day)', 'Recycling Rate (%)', 'Population Density (People/km²)', 'Municipal Efficiency Score (1-10)', 'Disposal Method', 'Cost of Waste Management (₹/Ton)', 'Awareness Campaigns Count', 'Landfill Name', 'Landfill Location (Lat, Long)', 'Landfill Capacity (Tons)', 'Year']

First few rows:


,City/District,Waste Type,Waste Generated (Tons/Day),Recycling Rate (%),Population Density (People/km²),Municipal Efficiency Score (1-10),Disposal Method,Cost of Waste Management (₹/Ton),Awareness Campaigns Count,Landfill Name,"Landfill Location (Lat, Long)",Landfill Capacity (Tons),Year
0,Mumbai,Plastic,6610,68,11191,9,Composting,3056,14,Mumbai Landfill,"22.4265, 77.4931",45575,2019
1,Mumbai,Organic,1181,56,11191,5,Composting,2778,12,Mumbai Landfill,"22.4265, 77.4931",45575,2019
2,Mumbai,E-Waste,8162,53,11191,8,Incineration,3390,13,Mumbai Landfill,"22.4265, 77.4931",45575,2019
3,Mumbai,Construction,8929,56,11191,5,Landfill,1498,14,Mumbai Landfill,"22.4265, 77.4931",45575,2019
4,Mumbai,Hazardous,5032,44,11191,7,Recycling,2221,16,Mumbai Landfill,"22.4265, 77.4931",45575,2019


In [3]:
# Drop City/District column
df_processed = df.drop(columns=['City/District'])

print(f"Shape after dropping City/District: {df_processed.shape}")
df_processed.head()


Shape after dropping City/District: (850, 12)


,Waste Type,Waste Generated (Tons/Day),Recycling Rate (%),Population Density (People/km²),Municipal Efficiency Score (1-10),Disposal Method,Cost of Waste Management (₹/Ton),Awareness Campaigns Count,Landfill Name,"Landfill Location (Lat, Long)",Landfill Capacity (Tons),Year
0,Plastic,6610,68,11191,9,Composting,3056,14,Mumbai Landfill,"22.4265, 77.4931",45575,2019
1,Organic,1181,56,11191,5,Composting,2778,12,Mumbai Landfill,"22.4265, 77.4931",45575,2019
2,E-Waste,8162,53,11191,8,Incineration,3390,13,Mumbai Landfill,"22.4265, 77.4931",45575,2019
3,Construction,8929,56,11191,5,Landfill,1498,14,Mumbai Landfill,"22.4265, 77.4931",45575,2019
4,Hazardous,5032,44,11191,7,Recycling,2221,16,Mumbai Landfill,"22.4265, 77.4931",45575,2019


In [4]:
# Split "Landfill Location (Lat, Long)" into separate Latitude and Longitude columns
# The location is stored as a string like "22.4265, 77.4931"
location_data = df_processed['Landfill Location (Lat, Long)'].str.split(', ', expand=True)
df_processed['Latitude'] = location_data[0].astype(float)
df_processed['Longitude'] = location_data[1].astype(float)

# Drop the original location column
df_processed = df_processed.drop(columns=['Landfill Location (Lat, Long)'])

print("Location column split into Latitude and Longitude")
print(f"\nNew columns: {df_processed.columns.tolist()}")
df_processed[['Latitude', 'Longitude']].head()


Location column split into Latitude and Longitude

New columns: ['Waste Type', 'Waste Generated (Tons/Day)', 'Recycling Rate (%)', 'Population Density (People/km²)', 'Municipal Efficiency Score (1-10)', 'Disposal Method', 'Cost of Waste Management (₹/Ton)', 'Awareness Campaigns Count', 'Landfill Name', 'Landfill Capacity (Tons)', 'Year', 'Latitude', 'Longitude']


,Latitude,Longitude
0,22.4265,77.4931
1,22.4265,77.4931
2,22.4265,77.4931
3,22.4265,77.4931
4,22.4265,77.4931


## Step 2: Feature Encoding


In [5]:
# Identify categorical and numeric columns
categorical_cols = ['Waste Type', 'Disposal Method', 'Landfill Name']
numeric_cols = [col for col in df_processed.columns if col not in categorical_cols]

print(f"Categorical columns: {categorical_cols}")
print(f"\nNumeric columns: {numeric_cols}")
print(f"\nData types:\n{df_processed.dtypes}")


Categorical columns: ['Waste Type', 'Disposal Method', 'Landfill Name']

Numeric columns: ['Waste Generated (Tons/Day)', 'Recycling Rate (%)', 'Population Density (People/km²)', 'Municipal Efficiency Score (1-10)', 'Cost of Waste Management (₹/Ton)', 'Awareness Campaigns Count', 'Landfill Capacity (Tons)', 'Year', 'Latitude', 'Longitude']

Data types:
Waste Type                            object
Waste Generated (Tons/Day)             int64
Recycling Rate (%)                     int64
Population Density (People/km²)        int64
Municipal Efficiency Score (1-10)      int64
Disposal Method                       object
Cost of Waste Management (₹/Ton)       int64
Awareness Campaigns Count              int64
Landfill Name                         object
Landfill Capacity (Tons)               int64
Year                                   int64
Latitude                             float64
Longitude                            float64
dtype: object


In [6]:
# Create a copy for encoding
df_encoded = df_processed.copy()

# Apply label encoding to categorical features
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_processed[col])
    label_encoders[col] = le
    print(f"{col}: {len(le.classes_)} unique values")
    print(f"  Classes: {le.classes_[:5]}..." if len(le.classes_) > 5 else f"  Classes: {le.classes_}")

print(f"\nEncoded data shape: {df_encoded.shape}")
df_encoded.head()


Waste Type: 5 unique values
  Classes: ['Construction' 'E-Waste' 'Hazardous' 'Organic' 'Plastic']
Disposal Method: 4 unique values
  Classes: ['Composting' 'Incineration' 'Landfill' 'Recycling']
Landfill Name: 34 unique values
  Classes: ['Agra Landfill' 'Ahmedabad Landfill' 'Allahabad Landfill'
 'Amritsar Landfill' 'Bengaluru Landfill']...

Encoded data shape: (850, 13)


,Waste Type,Waste Generated (Tons/Day),Recycling Rate (%),Population Density (People/km²),Municipal Efficiency Score (1-10),Disposal Method,Cost of Waste Management (₹/Ton),Awareness Campaigns Count,Landfill Name,Landfill Capacity (Tons),Year,Latitude,Longitude
0,4,6610,68,11191,9,0,3056,14,22,45575,2019,22.4265,77.4931
1,3,1181,56,11191,5,0,2778,12,22,45575,2019,22.4265,77.4931
2,1,8162,53,11191,8,1,3390,13,22,45575,2019,22.4265,77.4931
3,0,8929,56,11191,5,2,1498,14,22,45575,2019,22.4265,77.4931
4,2,5032,44,11191,7,3,2221,16,22,45575,2019,22.4265,77.4931


In [7]:
# Check for any missing values
print("Missing values:")
print(df_encoded.isnull().sum())
print(f"\nTotal missing values: {df_encoded.isnull().sum().sum()}")

# Prepare feature matrix X (all columns are now numeric)
X = df_encoded.values
print(f"\nFeature matrix X shape: {X.shape}")
print(f"Data type: {X.dtype}")


Missing values:
Waste Type                           0
Waste Generated (Tons/Day)           0
Recycling Rate (%)                   0
Population Density (People/km²)      0
Municipal Efficiency Score (1-10)    0
Disposal Method                      0
Cost of Waste Management (₹/Ton)     0
Awareness Campaigns Count            0
Landfill Name                        0
Landfill Capacity (Tons)             0
Year                                 0
Latitude                             0
Longitude                            0
dtype: int64

Total missing values: 0

Feature matrix X shape: (850, 13)
Data type: float64


## Step 3: K-Means Clustering


In [8]:
# Standardize features before clustering
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Scaled feature matrix shape: {X_scaled.shape}")
print(f"Mean of scaled features: {X_scaled.mean(axis=0)[:5]}...")
print(f"Std of scaled features: {X_scaled.std(axis=0)[:5]}...")


Scaled feature matrix shape: (850, 13)
Mean of scaled features: [ 0.00000000e+00  8.88178420e-17 -8.77729262e-17  6.68746104e-17
 -1.77635684e-16]...
Std of scaled features: [1. 1. 1. 1. 1.]...


In [9]:
# Apply K-means clustering with 3 clusters
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X)

print(f"Cluster labels shape: {cluster_labels.shape}")
print(f"Unique clusters: {np.unique(cluster_labels)}")
print(f"\nCluster distribution:")
unique, counts = np.unique(cluster_labels, return_counts=True)
for cluster, count in zip(unique, counts):
    print(f"  Cluster {cluster}: {count} samples ({count/len(cluster_labels)*100:.2f}%)")


Cluster labels shape: (850,)
Unique clusters: [0 1 2]

Cluster distribution:
  Cluster 0: 400 samples (47.06%)
  Cluster 1: 275 samples (32.35%)
  Cluster 2: 175 samples (20.59%)


## Step 4: Train-Test Split


In [10]:
# Split data into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, 
    cluster_labels, 
    test_size=0.2, 
    random_state=42,
    stratify=cluster_labels
)

print(f"Training set - X shape: {X_train.shape}, y shape: {y_train.shape}")
print(f"Testing set - X shape: {X_test.shape}, y shape: {y_test.shape}")

print(f"\nTraining set cluster distribution:")
unique_train, counts_train = np.unique(y_train, return_counts=True)
for cluster, count in zip(unique_train, counts_train):
    print(f"  Cluster {cluster}: {count} samples ({count/len(y_train)*100:.2f}%)")

print(f"\nTesting set cluster distribution:")
unique_test, counts_test = np.unique(y_test, return_counts=True)
for cluster, count in zip(unique_test, counts_test):
    print(f"  Cluster {cluster}: {count} samples ({count/len(y_test)*100:.2f}%)")


Training set - X shape: (680, 13), y shape: (680,)
Testing set - X shape: (170, 13), y shape: (170,)

Training set cluster distribution:
  Cluster 0: 320 samples (47.06%)
  Cluster 1: 220 samples (32.35%)
  Cluster 2: 140 samples (20.59%)

Testing set cluster distribution:
  Cluster 0: 80 samples (47.06%)
  Cluster 1: 55 samples (32.35%)
  Cluster 2: 35 samples (20.59%)


## Step 5: Save Training Data


In [11]:
# Save training features as X.npy
np.save('X.npy', X_train)
print(f"Saved X_train to X.npy")
print(f"  Shape: {X_train.shape}")
print(f"  Data type: {X_train.dtype}")

# Save training cluster labels as y.npy
np.save('y.npy', y_train)
print(f"\nSaved y_train to y.npy")
print(f"  Shape: {y_train.shape}")
print(f"  Data type: {y_train.dtype}")
print(f"  Unique labels: {np.unique(y_train)}")


Saved X_train to X.npy
  Shape: (680, 13)
  Data type: float64

Saved y_train to y.npy
  Shape: (680,)
  Data type: int32
  Unique labels: [0 1 2]


In [12]:
# Verify the saved files
X_loaded = np.load('X.npy')
y_loaded = np.load('y.npy')

print("Verification:")
print(f"X.npy shape: {X_loaded.shape}, matches X_train: {np.array_equal(X_loaded, X_train)}")
print(f"y.npy shape: {y_loaded.shape}, matches y_train: {np.array_equal(y_loaded, y_train)}")
print("\nFiles saved successfully!")


Verification:
X.npy shape: (680, 13), matches X_train: True
y.npy shape: (680,), matches y_train: True

Files saved successfully!
